In [43]:
!jupyter nbconvert --to notebook --execute preprocess.ipynb --inplace

[NbConvertApp] Converting notebook preprocess.ipynb to notebook
[NbConvertApp] Writing 118580 bytes to preprocess.ipynb


In [44]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pulp import *
from scipy.stats import poisson, norm
from pulp import LpProblem, LpMaximize, LpVariable, LpStatus, lpSum, value, PULP_CBC_CMD

In [45]:
OUT_CSV = "fantasy_enriched.csv"



| Position        | Point Source         | Points | Description                                                                 |
| --------------- | -------------------- | -----: | --------------------------------------------------------------------------- |
| **All Players** | Appearance (<60 min) |     +1 | Plays any minutes up to 59:59                                               |
| **All Players** | Appearance (60+ min) |     +1 | Additional point for reaching 60+ minutes (total appearance = **2 points**) |
| **All Players** | Assist               |     +3 | Provides an assist                                                          |
| **All Players** | Yellow Card          |     -1 | Receives a yellow card                                                      |
| **All Players** | Red Card             |     -2 | Receives a red card                                                         |
| **All Players** | Own Goal             |     -2 | Scores an own goal                                                          |
| **All Players** | Winning a Penalty    |     +2 | Wins a penalty for their team                                               |
| **All Players** | Conceding a Penalty  |     -1 | Commits a foul leading to a penalty                                         |

| Position       | Point Source                  | Points | Description                                        |
| -------------- | ----------------------------- | -----: | -------------------------------------------------- |
| **Goalkeeper** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet     |
| **Goalkeeper** | First Goal Conceded           |      0 | No deduction for conceding the first goal          |
| **Goalkeeper** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first                |
| **Goalkeeper** | Goal Scored                   |     +9 | Scores a goal                                      |
| **Goalkeeper** | Penalty Save                  |     +3 | Saves a penalty during normal play (not shootouts) |
| **Goalkeeper** | Every 3 Saves                 |     +1 | Earns 1 point for every 3 saves                    |

| Position     | Point Source                  | Points | Description                                    |
| ------------ | ----------------------------- | -----: | ---------------------------------------------- |
| **Defender** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet |
| **Defender** | First Goal Conceded           |      0 | No deduction for conceding the first goal      |
| **Defender** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first            |
| **Defender** | Goal Scored                   |     +7 | Scores a goal                                  |

| Position       | Point Source                | Points | Description                                    |
| -------------- | --------------------------- | -----: | ---------------------------------------------- |
| **Midfielder** | Clean Sheet                 |     +1 | Plays 60+ minutes and team keeps a clean sheet |
| **Midfielder** | Goal Scored                 |     +6 | Scores a goal                                  |
| **Midfielder** | Every 3 Tackles             |     +1 | Earns 1 point for every 3 tackles              |
| **Midfielder** | Every 2 Big Chances Created |     +1 | Earns 1 point for every 2 big chances created  |

| Position    | Point Source            | Points | Description                               |
| ----------- | ----------------------- | -----: | ----------------------------------------- |
| **Forward** | Goal Scored             |     +5 | Scores a goal                             |
| **Forward** | Every 2 Shots on Target |     +1 | Earns 1 point for every 2 shots on target |

| Position                | Point Source          | Points | Description                                                                                                                                                                        |
| ----------------------- | --------------------- | -----: | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Bonus (All Players)** | Direct Free-Kick Goal |     +1 | Additional bonus for scoring directly from a free kick (on top of goal points)                                                                                                     |
| **Bonus (All Players)** | Scouting Bonus        |     +2 | Scores **more than 4 base fantasy points** in a match **and** is selected by **<5%** of fantasy teams. Captaincy and booster points do **not** count toward the 4-point threshold. |



## Core identity / player info

| Column | Type | Description |
|---|---|---|
| `fifa_id` | int | Unique FIFA fantasy player ID |
| `name` | str | Player name |
| `squad_id` | int | Internal squad/nation ID |
| `team` | str | National team |
| `position` | str | DEF / MID / FWD / GK |
| `price` | float | Fantasy price |
| `status` | str | Availability status |
| `total_points` | int | Total tournament points |
| `avg_points` | float | Average points per match |
| `matches_played` | int | Matches with participation |
| `percent_selected` | float | Selection rate (%) |
| `next_fixture` | int | Next match ID |


## Round fantasy points

| Column | Type | Description |
|---|---|---|
| `round_1_points` | int | Points in round 1 |
| `round_2_points` | int | Points in round 2 |
| `round_3_points` | int | Points in round 3 |
| `round_4_points` | int | Points in round 4 |


## Betting / scorer mapping

| Column | Type | Description |
|---|---|---|
| `betano_matched_name` | str | Matched Betano market name |
| `betano_match_score` | float | Name matching confidence (0–100) |
| `anytime_scorer_prob` | float | Probability player scores anytime |
| `first_scorer_prob` | float | Probability scores first goal |
| `last_scorer_prob` | float | Probability scores last goal |


## Match context

| Column | Type | Description |
|---|---|---|
| `match_home` | str | Home team |
| `match_away` | str | Away team |
| `is_home` | bool | Player team is home |
| `opponent` | str | Opponent team |


## Match probabilities 

| Column | Type | Description |
|---|---|---|
| `match_home_win_prob` | float | Home win probability |
| `match_draw_prob` | float | Draw probability |
| `match_away_win_prob` | float | Away win probability |
| `match_btts_prob` | float | Both teams score probability |
| `match_over_25_prob` | float | Over 2.5 goals probability |
| `match_over_35_prob` | float | Over 3.5 goals probability |
| `team_score_prob` | float | Team scores ≥1 goal probability |
| `team_score_2_prob` | float | Team scores ≥2 goals probability |
| `team_cs_prob` | float | Team clean sheet probability |
| `opp_score_prob` | float | Opponent scores ≥1 goal probability |
| `opp_cs_prob` | float | Opponent clean sheet probability |
| `team_over_05_prob` | float | Team scores ≥1 goal probability |
| `team_over_15_prob` | float | Team scores ≥2 goals probability |
| `team_qualify_prob` | float | Team advances probability |
| `team_win2_prob` | float | Team wins by 2+ goals probability |
| `opp_over_05_prob` | float | Opponent scores ≥1 goal probability |


## Raw cumulative stats

| Column | Type | Description |
|---|---|---|
| `stat_GS` | int | Goals |
| `stat_AS` | int | Assists |
| `stat_CS` | int | Clean sheets |
| `stat_GC` | int | Goals conceded |
| `stat_MP` | int | Minutes played |
| `stat_YC` | int | Yellow cards |
| `stat_RC` | int | Red cards |
| `stat_ST` | int | Shots on target |
| `stat_SB` | int | Shots blocked |
| `stat_CC` | int | Chances created |
| `stat_PS` | int | Penalties saved |
| `stat_T` | int | Tackles |
| `stat_S` | int | Saves |
| `stat_SXI` | int | Starts (XI appearances) |
| `stat_OG` | int | Own goals |
| `stat_PC` | int | Penalties committed |
| `stat_PW` | int | Penalties won |
| `stat_FK` | int | Free kicks won |



## Round-by-round stats (1–4)

| Pattern | Type | Description |
|---|---|---|
| `round_i_SXI` | int | Started (0/1) |
| `round_i_MP` | int | Minutes |
| `round_i_AS` | int | Assists |
| `round_i_YC` | int | Yellow cards |
| `round_i_RC` | int | Red cards |
| `round_i_OG` | int | Own goals |
| `round_i_PW` | int | Penalties won |
| `round_i_PC` | int | Penalties committed |
| `round_i_CS` | int | Clean sheets |
| `round_i_GS` | int | Goals |
| `round_i_GC` | int | Goals conceded |
| `round_i_PS` | int | Penalties saved |
| `round_i_T` | int | Tackles |
| `round_i_CC` | int | Chances created |
| `round_i_ST` | int | Shots on target |
| `round_i_FK` | int | Free kicks |
| `round_i_S` | int | Saves |
| `round_i_SB` | int | Shots blocked |



## External matching / enrichment

| Column | Type | Description |
|---|---|---|
| `clubstats_matched_player` | str | External player match |
| `clubs_match_dist` | float | Matching distance score |



## Form / usage trends

| Column | Type | Description |
|---|---|---|
| `started_last_match` | int | Started last match |
| `starts_last_3` | int | Starts last 3 matches |
| `starts_last_5` | int | Starts last 5 matches |
| `start_rate` | float | Start frequency |
| `minutes_last_3_avg` | float | Avg minutes last 3 |
| `minutes_last_5_avg` | float | Avg minutes last 5 |



## Performance rates

| Column | Type | Description |
|---|---|---|
| `goal_rate` | float | Goals per minute |
| `goals_per_start` | float | Goals per start |
| `shots_on_target_per90` | float | SOT per 90 |
| `goals_per_shot_on_target` | float | Conversion rate |
| `assist_rate` | float | Assists per minute |
| `chance_created_per90` | float | Chances per 90 |
| `tackles_per90` | float | Tackles per 90 |
| `cc_per90` | float | Chances created per 90 |
| `sot_per90` | float | Shots on target per 90 |
| `saves_per90` | float | Saves per 90 |
| `yc_per90` | float | Yellow cards per 90 |
| `rc_per90` | float | Red cards per 90 |
| `penalty_conceded_rate` | float | Penalties conceded per 90 |
| `own_goal_rate` | float | Own goals per 90 |
| `penalties_won_per90` | float | Penalties won per 90 |



## Recent / streak features

| Column | Type | Description |
|---|---|---|
| `recent_goals_last3` | int | Goals last 3 matches |
| `goal_streak` | int | Consecutive scoring matches |
| `recent_assists_last3` | int | Assists last 3 matches |
| `recent_chances_created_per90` | float | Recent creation rate |
| `recent_cs_rate` | float | Clean sheet rate |
| `recent_tackles_per90` | float | Recent tackles |
| `recent_cc_per90` | float | Recent chances created |
| `recent_sot_per90` | float | Recent shots on target |
| `recent_saves_per90` | float | Recent saves |
| `recent_penalties_won` | int | Penalties won last 3 |



## Expected value features

| Column | Type | Description |
|---|---|---|
| `goal_expectation` | float | Scoring proxy |
| `goal_expectation2` | float | Scoring proxy (2+ goals) |
| `assist_expectation` | float | Assist proxy |
| `cs_expectation` | float | Clean sheet proxy |
| `expected_minutes` | float | Expected minutes |
| `expected_gc_penalty` | float | Defensive penalty |
| `expected_tackle_points` | float | Tackles contribution |
| `expected_cc_points` | float | Chance creation points |
| `expected_sot_points` | float | Shot contribution points |
| `expected_save_points` | float | Save contribution points |



## Fantasy points dynamics

| Column | Type | Description |
|---|---|---|
| `points_last1` | int | Last match points |
| `points_last3_avg` | float | Avg last 3 |
| `points_last5_avg` | float | Avg last 5 |
| `weighted_points` | float | Recency weighted |
| `rolling_std_points` | float | Variance last 5 |
| `rolling_max_points` | int | Max last 5 |



## Team-position ranking features

| Column | Type | Description |
|---|---|---|
| `price_rank_team_position` | float | Price rank |
| `selected_rank_team_position` | float | Selection rank |
| `points_rank_team_position` | float | Points rank |
| `minutes_rank_team_position` | float | Minutes rank |
| `starts_rank_team_position` | float | Starts rank |
| `anytime_rank_team_position` | float | Scoring odds rank |
| `shots_rank_team_position` | float | Shooting rank |
| `cc_rank_team_position` | float | Chance creation rank |
| `tackles_rank_team_position` | float | Tackles rank |

In [46]:
df= pd.read_csv("fantasy_enriched.csv")

In [47]:
df = df[df["status"] == "playing"].copy()

In [48]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [49]:
# probability of playing a minute

raw_any = (
    0.15 * df["minutes_rank_team_position"]
  + 0.20 * df["stat_MP"]        # minutes needs to be very team stratified since while for other point sources players are all eligible
  + 0.10 * df["starts_rank_team_position"]
  + 0.15 * df["price_rank_team_position"]     # for minutes its a team stratified signal.
  + 0.20 * df["selected_rank_team_position"]
  + 0.15 * df["minutes_last_3_avg"]
  + 0.05 * df["anytime_rank_team_position"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_any"] = sigmoid((raw_any - 0.5) * 6)

In [50]:
df.loc[
    :,
    [
        "prob_plays_any",
        "name",
        "team",
        "position",
        "stat_MP",
        "minutes_rank_team_position",
        "starts_rank_team_position",
        "selected_rank_team_position",
        "price_rank_team_position",
        "anytime_rank_team_position",
    ],
].sort_values("prob_plays_any", ascending=False).head(30)

,prob_plays_any,name,team,position,stat_MP,minutes_rank_team_position,starts_rank_team_position,selected_rank_team_position,price_rank_team_position,anytime_rank_team_position
10,0.952574,Emiliano Martínez,Argentina,GK,1.000000,1.000000,1.000000,1.0,1.000000,1.000000
342,0.947350,Thibaut Courtois,Belgium,GK,1.000000,1.000000,1.000000,1.0,1.000000,0.633333
395,0.947350,Yassine Bounou,Morocco,GK,1.000000,1.000000,1.000000,1.0,1.000000,0.633333
149,0.940643,Kylian Mbappé,France,FWD,0.900000,1.000000,1.000000,1.0,1.000000,1.000000
134,0.937479,Jordan Pickford,England,GK,0.923077,1.000000,1.000000,1.0,1.000000,0.633333
70,0.937479,Maxime Crépeau,Canada,GK,0.923077,1.000000,1.000000,1.0,1.000000,0.633333
91,0.937479,Camilo Vargas,Colombia,GK,0.923077,1.000000,1.000000,1.0,1.000000,0.633333
348,0.937479,Diogo Costa,Portugal,GK,0.923077,1.000000,1.000000,1.0,1.000000,0.633333
289,0.937479,Gregor Kobel,Switzerland,GK,0.923077,1.000000,1.000000,1.0,1.000000,0.633333
153,0.937479,Mike Maignan,France,GK,0.923077,1.000000,1.000000,1.0,1.000000,0.633333


In [51]:
# probability of starting / playing 60 minutes

raw_any = (
    0.15 * df["minutes_rank_team_position"]
  + 0.10 * df["starts_rank_team_position"]
  + 0.25 * df["stat_MP"]
  + 0.10 * df["price_rank_team_position"]
  + 0.05 * df["selected_rank_team_position"]
  + 0.15 * df["minutes_last_3_avg"]
  + 0.15 * df["round_4_MP"]
  + 0.05 * df["anytime_rank_team_position"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_60min"] = sigmoid((raw_any - 0.5) * 6)


In [52]:
cols = [
    "name",
    "team",
    "position",
    "prob_plays_60min",
    "minutes_rank_team_position",
    "starts_rank_team_position",
    "price_rank_team_position",
    "selected_rank_team_position",
    "minutes_last_3_avg",
    "round_4_MP",
    "anytime_rank_team_position",
]

df[cols].sort_values("prob_plays_60min", ascending=False).head(30)

,name,team,position,prob_plays_60min,minutes_rank_team_position,starts_rank_team_position,price_rank_team_position,selected_rank_team_position,minutes_last_3_avg,round_4_MP,anytime_rank_team_position
10,Emiliano Martínez,Argentina,GK,0.952574,1.000000,1.000000,1.000000,1.0000,1.000000,1.000000,1.000000
342,Thibaut Courtois,Belgium,GK,0.947350,1.000000,1.000000,1.000000,1.0000,1.000000,1.000000,0.633333
395,Yassine Bounou,Morocco,GK,0.947350,1.000000,1.000000,1.000000,1.0000,1.000000,1.000000,0.633333
212,Gustavo Gómez,Paraguay,DEF,0.926121,1.000000,0.214286,1.000000,1.0000,1.000000,1.000000,1.000000
396,Achraf Hakimi,Morocco,DEF,0.923039,1.000000,0.140625,1.000000,1.0000,1.000000,1.000000,1.000000
149,Kylian Mbappé,France,FWD,0.922048,1.000000,1.000000,1.000000,1.0000,0.870000,0.708333,1.000000
289,Gregor Kobel,Switzerland,GK,0.921262,1.000000,1.000000,1.000000,1.0000,0.900000,0.750000,0.633333
134,Jordan Pickford,England,GK,0.921262,1.000000,1.000000,1.000000,1.0000,0.900000,0.750000,0.633333
153,Mike Maignan,France,GK,0.921262,1.000000,1.000000,1.000000,1.0000,0.900000,0.750000,0.633333
70,Maxime Crépeau,Canada,GK,0.921262,1.000000,1.000000,1.000000,1.0000,0.900000,0.750000,0.633333


There is a phenomenon where a player will be overvalued, such as Lukaku here for belgium. He has not played much, but since Belgium only has 3 forward and only him has significative minutes he ranks number 1 for all ranks in BEL,FWD. Need to include more general sums and beware with manual selection.

In [53]:
raw_goal = (
    0.60 * df["anytime_scorer_prob"]
  + 0.05 * df["recent_goals_last3"]
  + 0.05 * df["team_score_2_prob"]
  + 0.02 * df["stat_GS"]
  + 0.03 * df["stat_ST"]
  + 0.05 * df["price"]
)

df["prob_scores"] = raw_goal * df["prob_plays_60min"]

In [54]:
cols = [
    "name",
    "team",
    "position",
    "prob_scores",
    "goal_expectation",
    "anytime_scorer_prob",
    "recent_goals_last3",
    "recent_sot_per90",
    "goal_rate",
    "shots_on_target_per90",
    "stat_GS",
    "stat_ST",
    "price_rank_team_position",
]

df[cols].sort_values("prob_scores", ascending=False).head(30)

,name,team,position,prob_scores,goal_expectation,anytime_scorer_prob,recent_goals_last3,recent_sot_per90,goal_rate,shots_on_target_per90,stat_GS,stat_ST,price_rank_team_position
149,Kylian Mbappé,France,FWD,0.731316,1.000000,1.000000,1.00,0.034483,0.324786,0.351852,0.857143,0.866667,1.000000
5,Lionel Messi,Argentina,FWD,0.686795,0.892827,0.943886,1.00,0.045833,0.415625,0.445312,1.000000,1.000000,1.000000
48,Vinícius Júnior,Brazil,MID,0.507463,0.661569,0.718813,0.75,0.034483,0.216524,0.270655,0.571429,0.666667,1.000000
262,Mikel Oyarzabal,Spain,FWD,0.505895,0.648677,0.725769,1.00,0.033175,0.252492,0.252492,0.571429,0.533333,1.000000
199,Erling Haaland,Norway,FWD,0.486610,0.552767,0.725769,0.75,0.027778,0.351852,0.316667,0.714286,0.600000,1.000000
128,Harry Kane,England,FWD,0.473286,0.500675,0.633792,0.75,0.022727,0.268362,0.241525,0.714286,0.600000,1.000000
150,Ousmane Dembélé,France,MID,0.471469,0.727227,0.725769,1.00,0.024038,0.263889,0.164931,0.571429,0.333333,1.000000
352,Cristiano Ronaldo,Portugal,FWD,0.424944,0.445133,0.591127,0.75,0.026820,0.162393,0.189459,0.428571,0.466667,1.000000
31,Romelu Lukaku,Belgium,FWD,0.407883,0.602884,0.698717,0.50,0.013072,0.214689,0.107345,0.285714,0.133333,1.000000
49,Matheus Cunha,Brazil,FWD,0.395733,0.583849,0.633792,0.75,0.024272,0.242553,0.202128,0.428571,0.333333,1.000000


In [55]:
prob_assists = (
    0.05 * df["chance_created_per90"]
  + 0.20 * df["stat_CC"]
  + 0.05 * df["recent_cc_per90"]
  + 0.15 * df["team_score_2_prob"]
  + 0.15 * df["recent_assists_last3"]
  + 0.15 * df["price"]
  + 0.10 * df["stat_AS"]
)

df["expected_assists"] = prob_assists * df["prob_plays_60min"]

In [56]:
cols = [
    "name",
    "team",
    "position",
    "expected_assists",
    "assist_expectation",
    "stat_CC",
    "recent_cc_per90",
    "team_score_2_prob",
    "recent_assists_last3",
    "price",
    "prob_plays_60min",
    "stat_AS",
]

df[cols].sort_values("expected_assists", ascending=False).head(30)

,name,team,position,expected_assists,assist_expectation,stat_CC,recent_cc_per90,team_score_2_prob,recent_assists_last3,price,prob_plays_60min,stat_AS
162,Michael Olise,France,MID,0.599446,0.194970,1.0,0.178899,1.000000,1.00,0.857143,0.803942,1.0
149,Kylian Mbappé,France,FWD,0.423141,0.034217,0.2,0.049808,1.000000,0.50,1.000000,0.922048,0.4
5,Lionel Messi,Argentina,FWD,0.397895,0.142004,0.8,0.216667,0.797880,0.00,0.928571,0.912498,0.0
323,Bruno Guimarães,Brazil,MID,0.397142,0.097390,0.6,0.150000,0.688782,0.75,0.471429,0.796764,0.8
258,Marc Cucurella,Spain,DEF,0.359103,0.089348,0.6,0.096296,0.628933,0.75,0.228571,0.834914,0.6
150,Ousmane Dembélé,France,MID,0.350990,0.041702,0.2,0.062500,1.000000,0.50,0.928571,0.781460,0.4
286,Breel Embolo,Switzerland,FWD,0.328416,0.074975,0.6,0.050584,0.274282,0.50,0.571429,0.890720,0.4
48,Vinícius Júnior,Brazil,MID,0.300433,0.031446,0.2,0.049808,0.688782,0.25,0.928571,0.873305,0.2
172,Roberto Alvarado,Mexico,FWD,0.296067,0.103965,0.8,0.156000,0.280267,0.50,0.257143,0.760523,0.6
141,Jude Bellingham,England,MID,0.290268,0.090558,0.6,0.166667,0.392613,0.25,0.685714,0.823783,0.2


In [57]:
position_yc_modifier = {
    "GK": 0.15,
    "FWD": 0.35,
    "DEF": 0.45,
    "MID": 0.55,
}

df["position_yc_modifier"] = df["position"].map(position_yc_modifier)

raw_yc = (
    0.35 * df["yc_per90"]
  + 0.15 * df["tackles_per90"]
  + 0.15 * df["opp_over_05_prob"]
  + 0.05 * df["rc_per90"]
  + 0.05 * df["position_yc_modifier"]
)

df["prob_yellow_card"] = sigmoid((raw_yc - 0.6) * 5) * df["prob_plays_60min"]

In [58]:
cols = [
    "name",
    "team",
    "position",
    "prob_yellow_card",
    "yc_per90",
    "stat_YC",
    "tackles_per90",
    "recent_tackles_per90",
    "opp_over_05_prob",
    "prob_plays_60min",
    "rc_per90",
]

df[cols].sort_values("prob_yellow_card", ascending=False).head(30)

,name,team,position,prob_yellow_card,yc_per90,stat_YC,tackles_per90,recent_tackles_per90,opp_over_05_prob,prob_plays_60min,rc_per90
210,Matías Galarza,Paraguay,MID,0.131623,0.255034,1.0,0.046980,0.234899,1.000000,0.804053,0.000000
112,Yasser Ibrahim,Egypt,DEF,0.112540,0.202128,1.0,0.018617,0.104895,0.895727,0.826859,0.000000
362,Andrés Cubas,Paraguay,MID,0.105619,0.097436,0.5,0.035897,0.150000,1.000000,0.822493,0.000000
219,Miguel Almirón,Paraguay,MID,0.098465,0.177570,0.5,0.032710,0.185185,1.000000,0.583723,0.728972
212,Gustavo Gómez,Paraguay,DEF,0.098385,0.000000,0.0,0.010256,0.050000,1.000000,0.926121,0.000000
213,Júnior Alonso,Paraguay,DEF,0.098055,0.126667,0.5,0.026667,0.142857,1.000000,0.750913,0.000000
217,Juan José Cáceres,Paraguay,DEF,0.097152,0.108883,0.5,0.051576,0.240741,1.000000,0.752093,0.000000
31,Romelu Lukaku,Belgium,FWD,0.094936,0.214689,0.5,0.000000,0.000000,0.704143,0.802544,0.000000
101,Marwan Attia,Egypt,MID,0.093430,0.110465,0.5,0.017442,0.059055,0.895727,0.773043,0.000000
228,Orlando Gill,Paraguay,GK,0.090798,0.000000,0.0,0.000000,0.000000,1.000000,0.920561,0.000000


In [59]:
raw_pw = (
    0.10 * df["stat_PW"]                    # tournament history
  + 0.25 * df["anytime_scorer_prob"]      # gets into dangerous areas
  + 0.10 * df["chance_created_per90"]     # attacking involvement
  + 0.15 * df["stat_CC"]                  # recent form in creation
  + 0.15 * df["team_score_2_prob"]        # team expected to attack heavily
  + 0.10 * df["price_rank_team_position"] # quality proxy
)

# Penalties are rare → heavily compress probabilities + minutes as external gate
df["prob_pen_won"] = 0.15 * sigmoid((raw_pw - 0.65) * 5) * df["prob_plays_60min"]

In [60]:
cols = [
    "name",
    "team",
    "position",
    "prob_pen_won",
    "stat_PW",
    "anytime_scorer_prob",
    "assist_expectation",
    "chance_created_per90",
    "team_score_2_prob",
    "price_rank_team_position",
    "prob_plays_60min",
]

df[cols].sort_values("prob_pen_won", ascending=False).head(30)

,name,team,position,prob_pen_won,stat_PW,anytime_scorer_prob,assist_expectation,chance_created_per90,team_score_2_prob,price_rank_team_position,prob_plays_60min
5,Lionel Messi,Argentina,FWD,0.057939,0.0,0.943886,0.142004,0.125000,0.797880,1.000000,0.912498
149,Kylian Mbappé,France,FWD,0.049460,0.0,1.000000,0.034217,0.028490,1.000000,1.000000,0.922048
162,Michael Olise,France,MID,0.044351,0.0,0.541660,0.194970,0.162338,1.000000,0.900000,0.803942
150,Ousmane Dembélé,France,MID,0.033273,0.0,0.725769,0.041702,0.034722,1.000000,1.000000,0.781460
320,Lautaro Martínez,Argentina,FWD,0.032755,1.0,0.692224,0.046750,0.041152,0.797880,0.725000,0.682138
48,Vinícius Júnior,Brazil,MID,0.031012,0.0,0.718813,0.031446,0.028490,0.688782,1.000000,0.873305
262,Mikel Oyarzabal,Spain,FWD,0.030328,0.0,0.725769,0.035620,0.033223,0.628933,1.000000,0.876473
186,Ismael Saibari,Morocco,MID,0.029273,0.0,0.579688,0.057923,0.055096,0.558311,1.000000,0.896428
286,Breel Embolo,Switzerland,FWD,0.025375,0.0,0.480445,0.074975,0.086455,0.274282,1.000000,0.890720
39,Youri Tielemans,Belgium,MID,0.025138,1.0,0.322152,0.026886,0.025974,0.528728,0.700000,0.873335


In [61]:
raw_cs = (
    0.8 * df["team_cs_prob"]            # strongest signal
  + 0.10 * df["stat_CS"]               # tournament history
  + 0.10 * (1 - df["stat_GC"])         # fewer goals conceded is better
)

base_cs = sigmoid((raw_cs - 0.5) * 6)

df["prob_clean_sheet"] = base_cs * df["prob_plays_60min"]

df.loc[df["position"] == "FWD", "prob_clean_sheet"] = 0.0

In [62]:
cols = [
    "name",
    "team",
    "position",
    "prob_clean_sheet",
    "team_cs_prob",
    "prob_plays_60min",
    "stat_CS",
    "stat_GC",
    "minutes_rank_team_position",
    "tackles_per90",
]

df[cols].sort_values("prob_clean_sheet", ascending=False).head(30)

,name,team,position,prob_clean_sheet,team_cs_prob,prob_plays_60min,stat_CS,stat_GC,minutes_rank_team_position,tackles_per90
153,Mike Maignan,France,GK,0.853200,1.000000,0.921262,0.50,0.285714,1.000000,0.000000
10,Emiliano Martínez,Argentina,GK,0.837330,0.904231,0.952574,0.50,0.428571,1.000000,0.000000
144,Dayot Upamecano,France,DEF,0.787890,1.000000,0.850742,0.50,0.285714,1.000000,0.020231
19,Alexis Mac Allister,Argentina,MID,0.767535,0.904231,0.850990,0.75,0.285714,1.000000,0.036474
162,Michael Olise,France,MID,0.744547,1.000000,0.803942,0.50,0.285714,1.000000,0.012987
332,Jules Koundé,France,DEF,0.743794,1.000000,0.803128,0.50,0.285714,0.877778,0.032836
150,Ousmane Dembélé,France,MID,0.735131,1.000000,0.781460,0.75,0.142857,0.900000,0.010417
20,Enzo Fernández,Argentina,MID,0.731602,0.904231,0.824023,0.50,0.285714,0.900000,0.020000
316,Lisandro Martínez,Argentina,DEF,0.714242,0.904231,0.804470,0.50,0.285714,1.000000,0.010000
0,Cristian Romero,Argentina,DEF,0.684211,0.904231,0.770646,0.50,0.285714,0.725000,0.019455


In [63]:
position_rc_modifier = {
    "GK": 0.04,
    "FWD": 0.08,
    "DEF": 0.12,
    "MID": 0.14,
}

df["position_rc_modifier"] = df["position"].map(position_rc_modifier)

raw_rc = (
    0.30 * df["position_rc_modifier"]   # strongest prior
  + 0.25 * df["rc_per90"]              # direct history
  + 0.15 * df["tackles_per90"]         # physical involvement
  + 0.15 * df["prob_yellow_card"]      # disciplinary tendency
  + 0.10 * df["opp_over_05_prob"]      # more defending -> more challenges
)

# Red cards are extremely rare → strong compression + external gate
df["prob_red_card"] = 0.05 * sigmoid((raw_rc - 0.5) * 5) * df["prob_plays_60min"]

In [64]:
cols = [
    "name",
    "team",
    "position",
    "prob_red_card",
    "position_rc_modifier",
    "rc_per90",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_red_card", ascending=False).head(30)

,name,team,position,prob_red_card,position_rc_modifier,rc_per90,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
219,Miguel Almirón,Paraguay,MID,0.009171,0.14,0.728972,0.032710,0.098465,1.000000,0.586649
165,César Montes,Mexico,DEF,0.007758,0.12,0.577778,0.007407,0.063807,0.593284,0.565440
212,Gustavo Gómez,Paraguay,DEF,0.006923,0.12,0.000000,0.010256,0.098385,1.000000,0.926121
210,Matías Galarza,Paraguay,MID,0.006444,0.14,0.000000,0.046980,0.131623,1.000000,0.785710
362,Andrés Cubas,Paraguay,MID,0.006439,0.14,0.000000,0.035897,0.105619,1.000000,0.678088
361,Mohamed Salah,Egypt,MID,0.006399,0.14,0.000000,0.002959,0.088875,0.895727,0.885897
108,Mohamed Hany,Egypt,DEF,0.006320,0.12,0.000000,0.035897,0.088192,0.895727,0.840154
228,Orlando Gill,Paraguay,GK,0.006137,0.04,0.000000,0.000000,0.090798,1.000000,0.902911
113,Omar Marmoush,Egypt,FWD,0.006034,0.08,0.000000,0.012658,0.086713,0.895727,0.909109
230,Julio Enciso,Paraguay,MID,0.006010,0.14,0.000000,0.012270,0.085833,1.000000,0.844803


In [65]:
position_og_modifier = {
    "GK": 0.01,
    "FWD": 0.02,
    "MID": 0.04,
    "DEF": 0.08,
}

df["position_og_modifier"] = df["position"].map(position_og_modifier)

raw_og = (
    0.4 * df["position_og_modifier"]
  + 0.25 * df["opp_over_05_prob"]
  + 0.10 * df["tackles_per90"]
  + 0.10 * (1 - df["stat_GC"])
)

df["prob_own_goal"] = 0.03 * sigmoid((raw_og - 0.5) * 5) * df["prob_plays_60min"]

In [66]:
cols = [
    "name",
    "team",
    "position",
    "prob_own_goal",
    "position_og_modifier",
    "own_goal_rate",
    "opp_over_05_prob",
    "tackles_per90",
    "stat_GC",
    "prob_plays_any",
]

df[cols].sort_values("prob_own_goal", ascending=False).head(30)

,name,team,position,prob_own_goal,position_og_modifier,own_goal_rate,opp_over_05_prob,tackles_per90,stat_GC,prob_plays_any
210,Matías Galarza,Paraguay,MID,0.007908,0.04,0.000000,1.000000,0.046980,0.142857,0.785710
212,Gustavo Gómez,Paraguay,DEF,0.007793,0.08,0.000000,1.000000,0.010256,0.714286,0.926121
108,Mohamed Hany,Egypt,DEF,0.007123,0.08,0.005128,0.895727,0.035897,0.571429,0.840154
233,Nuno Mendes,Portugal,DEF,0.007115,0.08,0.000000,0.802058,0.014663,0.285714,0.892531
112,Yasser Ibrahim,Egypt,DEF,0.007042,0.08,0.000000,0.895727,0.018617,0.428571,0.786872
228,Orlando Gill,Paraguay,GK,0.006964,0.01,0.000000,1.000000,0.000000,0.714286,0.902911
113,Omar Marmoush,Egypt,FWD,0.006962,0.02,0.000000,0.895727,0.012658,0.428571,0.909109
109,Ramy Rabia,Egypt,DEF,0.006856,0.08,0.000000,0.895727,0.014388,0.285714,0.648912
348,Diogo Costa,Portugal,GK,0.006798,0.01,0.000000,0.802058,0.000000,0.285714,0.937479
217,Juan José Cáceres,Paraguay,DEF,0.006756,0.08,0.000000,1.000000,0.051576,0.571429,0.610705


In [67]:
position_pc_modifier = {
    "GK": 0.02,
    "FWD": 0.03,
    "MID": 0.06,
    "DEF": 0.10,
}

df["position_pc_modifier"] = df["position"].map(position_pc_modifier)

raw_pc = (
    0.30 * df["position_pc_modifier"]
  + 0.25 * df["penalty_conceded_rate"]
  + 0.05 * df["tackles_per90"]
  + 0.15 * df["prob_yellow_card"]
  + 0.15 * df["opp_over_05_prob"]
)

df["prob_penalty_committed"] = 0.08 * sigmoid((raw_pc - 0.5) * 5) * df["prob_plays_60min"]

In [68]:
cols = [
    "name",
    "team",
    "position",
    "prob_penalty_committed",
    "position_pc_modifier",
    "penalty_conceded_rate",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_penalty_committed", ascending=False).head(30)

,name,team,position,prob_penalty_committed,position_pc_modifier,penalty_conceded_rate,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
212,Gustavo Gómez,Paraguay,DEF,0.013256,0.10,0.0,0.010256,0.098385,1.000000,0.926121
228,Orlando Gill,Paraguay,GK,0.011846,0.02,0.0,0.000000,0.090798,1.000000,0.902911
108,Mohamed Hany,Egypt,DEF,0.011732,0.10,0.0,0.035897,0.088192,0.895727,0.840154
362,Andrés Cubas,Paraguay,MID,0.011315,0.06,0.0,0.035897,0.105619,1.000000,0.678088
210,Matías Galarza,Paraguay,MID,0.011267,0.06,0.0,0.046980,0.131623,1.000000,0.785710
112,Yasser Ibrahim,Egypt,DEF,0.011212,0.10,0.0,0.018617,0.112540,0.895727,0.786872
361,Mohamed Salah,Egypt,MID,0.011168,0.06,0.0,0.002959,0.088875,0.895727,0.885897
113,Omar Marmoush,Egypt,FWD,0.010909,0.03,0.0,0.012658,0.086713,0.895727,0.909109
217,Juan José Cáceres,Paraguay,DEF,0.010849,0.10,0.0,0.051576,0.097152,1.000000,0.610705
233,Nuno Mendes,Portugal,DEF,0.010826,0.10,0.0,0.014663,0.080598,0.802058,0.892531


In [69]:
df["lambda_opp"] = -np.log(df["team_cs_prob"].clip(lower=0.01))

df["prob_gc_0"] = np.exp(-df["lambda_opp"])
df["prob_gc_1"] = df["lambda_opp"] * np.exp(-df["lambda_opp"])
df["prob_gc_2"] = (df["lambda_opp"] ** 2 / 2) * np.exp(-df["lambda_opp"])
df["prob_gc_3"] = (df["lambda_opp"] ** 3 / 6) * np.exp(-df["lambda_opp"])
df["prob_gc_4plus"] = 1 - (
    df["prob_gc_0"]
    + df["prob_gc_1"]
    + df["prob_gc_2"]
    + df["prob_gc_3"]
)

# RAW expected goals conceded (true intensity)
df["expected_goals_conceded"] = df["lambda_opp"]

# Normalize ONLY as auxiliary feature (separate column)
scaler = MinMaxScaler()
df["expected_goals_conceded_norm"] = scaler.fit_transform(
    df[["expected_goals_conceded"]]
)

# Player exposure to goals conceded (correct)
df["expected_player_goals_conceded"] = (
    df["lambda_opp"] * df["prob_plays_60min"]
)

# GC penalty MUST use raw λ (correct Poisson expectation)
df["expected_gc_penalty"] = (
    -(df["lambda_opp"] - 1 + np.exp(-df["lambda_opp"]))
    * df["prob_plays_60min"]
)

df.loc[df["position"].isin(["MID", "FWD"]), [
    "expected_player_goals_conceded",
    "expected_gc_penalty"
]] = 0.0

In [70]:
cols = [
    "name",
    "team",
    "position",
    "prob_plays_60min",
    "team_cs_prob",
    "lambda_opp",
    "prob_gc_0",
    "prob_gc_1",
    "prob_gc_2",
    "prob_gc_3",
    "prob_gc_4plus",
    "expected_player_goals_conceded",
    "expected_gc_penalty",
]

df[df["position"].isin(["GK", "DEF"])][cols] \
    .sort_values("expected_gc_penalty") \
    .head(30)

,name,team,position,prob_plays_60min,team_cs_prob,lambda_opp,prob_gc_0,prob_gc_1,prob_gc_2,prob_gc_3,prob_gc_4plus,expected_player_goals_conceded,expected_gc_penalty
212,Gustavo Gómez,Paraguay,DEF,0.926121,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,4.264943,-3.348084
228,Orlando Gill,Paraguay,GK,0.920561,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,4.239342,-3.327986
217,Juan José Cáceres,Paraguay,DEF,0.752093,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,3.463517,-2.718945
213,Júnior Alonso,Paraguay,DEF,0.750913,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,3.458083,-2.714679
214,Omar Alderete,Paraguay,DEF,0.461743,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,2.126403,-1.669278
218,José Canale,Paraguay,DEF,0.342546,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,1.577485,-1.238364
118,Mostafa Shobeir,Egypt,GK,0.901699,0.104868,2.255052,0.104868,0.236483,0.266641,0.200430,0.191579,2.033379,-1.226239
108,Mohamed Hany,Egypt,DEF,0.875242,0.104868,2.255052,0.104868,0.236483,0.266641,0.200430,0.191579,1.973717,-1.190259
112,Yasser Ibrahim,Egypt,DEF,0.826859,0.104868,2.255052,0.104868,0.236483,0.266641,0.200430,0.191579,1.864611,-1.124463
216,Gustavo Velázquez,Paraguay,DEF,0.310458,0.000000,4.605170,0.010000,0.046052,0.106038,0.162774,0.675136,1.429713,-1.122359


In [71]:
raw_save = (
    0.25 * df["expected_goals_conceded"]  # opportunity
  + 0.20 * df["saves_per90"]             # ability
  + 0.05 * df["recent_saves_per90"]      # recent form
  + 0.20 * df["price"]                   # overall GK quality
)

# Expected saves (λ)
df["expected_saves"] = (
    sigmoid((raw_save - 0.5) * 6) * 4
) * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "expected_saves"] = 0.0



In [72]:
cols = [
    "name",
    "team",
    "expected_saves",
    "expected_goals_conceded",
    "saves_per90",
    "recent_saves_per90",
    "price",
    "prob_plays_any",
]

df[df["position"] == "GK"][cols] \
    .sort_values("expected_saves", ascending=False) \
    .head(30)

,name,team,expected_saves,expected_goals_conceded,saves_per90,recent_saves_per90,price,prob_plays_any
228,Orlando Gill,Paraguay,3.662987,4.605170,0.876923,0.960000,0.000000,0.902911
118,Mostafa Shobeir,Egypt,2.680404,2.255052,0.461538,0.420000,0.000000,0.881843
348,Diogo Costa,Portugal,2.458976,1.570620,0.700000,0.866667,0.200000,0.937479
203,Ørjan Nyland,Norway,2.196057,1.851759,0.400000,0.600000,0.100000,0.896671
289,Gregor Kobel,Switzerland,1.727008,1.125822,0.650000,0.666667,0.171429,0.937479
342,Thibaut Courtois,Belgium,1.574623,1.196268,0.415385,0.420000,0.200000,0.947350
70,Maxime Crépeau,Canada,1.458121,1.420541,0.250000,0.200000,0.071429,0.937479
313,Matt Freese,USA,1.424732,1.317726,0.333333,0.500000,0.100000,0.896671
56,Alisson Becker,Brazil,1.170635,0.772659,0.550000,0.600000,0.214286,0.923274
204,Egil Selvik,Norway,1.146995,1.851759,1.000000,1.000000,0.042857,0.449405


In [73]:
raw_pen_save = (
    0.40 * df["price"]                    
  + 0.25 * df["expected_goals_conceded"]
  + 0.15 * df["opp_over_05_prob"]
  + 0.05 * df["stat_PS"]
)

# Strong compression because penalty saves are exceptionally rare
df["prob_penalty_save"] = sigmoid((raw_pen_save - 0.5) * 3) / 20 * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "prob_penalty_save"] = 0.0

In [74]:
cols = [
    "name",
    "team",
    "prob_penalty_save",
    "price",
    "expected_goals_conceded",
    "opp_over_05_prob",
    "prob_plays_60min",
    "stat_PS",
]

df[df["position"] == "GK"][cols] \
    .sort_values("prob_penalty_save", ascending=False) \
    .head(30)

,name,team,prob_penalty_save,price,expected_goals_conceded,opp_over_05_prob,prob_plays_60min,stat_PS
228,Orlando Gill,Paraguay,0.042213,0.000000,4.605170,1.000000,0.920561,0.0
118,Mostafa Shobeir,Egypt,0.030565,0.000000,2.255052,0.895727,0.901699,1.0
348,Diogo Costa,Portugal,0.026223,0.200000,1.570620,0.802058,0.921262,0.0
203,Ørjan Nyland,Norway,0.025743,0.100000,1.851759,0.847671,0.863358,0.0
342,Thibaut Courtois,Belgium,0.023140,0.200000,1.196268,0.704143,0.947350,0.0
70,Maxime Crépeau,Canada,0.022946,0.071429,1.420541,0.758707,0.921262,0.0
289,Gregor Kobel,Switzerland,0.021368,0.171429,1.125822,0.678150,0.921262,0.0
313,Matt Freese,USA,0.020907,0.100000,1.317726,0.730981,0.863358,0.0
56,Alisson Becker,Brazil,0.017996,0.214286,0.772659,0.537898,0.908430,0.0
266,Unai Simón,Spain,0.017717,0.214286,0.734007,0.527296,0.912904,0.0


In [75]:
raw_tackles = (
    0.40 * df["tackles_per90"]
  + 0.20 * df["recent_tackles_per90"]
  + 0.05 * df["expected_goals_conceded"]
  + 0.10 * df["match_over_25_prob"]
  + 0.15 * df["price"]
)

lam_tackles = sigmoid((raw_tackles - 0.5) * 6) * 3
df["expected_tackle_points"] = (lam_tackles / 3) * df["prob_plays_60min"]
df.loc[df["position"] != "MID", "expected_tackle_points"] = 0.0

In [76]:
cols = [
    "name",
    "team",
    "expected_tackle_points",
    "tackles_per90",
    "recent_tackles_per90",
    "expected_goals_conceded",
    "match_over_25_prob",
    "price",
    "prob_plays_60min",
]

df[df["position"] == "MID"][cols] \
    .sort_values("expected_tackle_points", ascending=False) \
    .head(30)

,name,team,expected_tackle_points,tackles_per90,recent_tackles_per90,expected_goals_conceded,match_over_25_prob,price,prob_plays_60min
210,Matías Galarza,Paraguay,0.311792,0.046980,0.234899,4.605170,1.000000,0.185714,0.804053
362,Andrés Cubas,Paraguay,0.291820,0.035897,0.150000,4.605170,1.000000,0.171429,0.822493
230,Julio Enciso,Paraguay,0.290604,0.012270,0.042373,4.605170,1.000000,0.442857,0.789073
219,Miguel Almirón,Paraguay,0.234775,0.032710,0.185185,4.605170,1.000000,0.357143,0.583723
361,Mohamed Salah,Egypt,0.204819,0.002959,0.000000,2.255052,0.475278,0.928571,0.881791
231,Diego Gómez,Paraguay,0.200562,0.033898,0.192308,4.605170,1.000000,0.471429,0.466473
229,Damián Bobadilla,Paraguay,0.178130,0.036842,0.241379,4.605170,1.000000,0.285714,0.439449
251,Bruno Fernandes,Portugal,0.171847,0.024024,0.102881,1.570620,0.614339,0.714286,0.825673
48,Vinícius Júnior,Brazil,0.170297,0.005698,0.019157,0.772659,0.796663,0.928571,0.873305
371,Martin Ødegaard,Norway,0.167691,0.022989,0.111111,1.851759,0.796663,0.600000,0.745991


In [77]:
raw_cc = (
    0.20 * df["cc_per90"]
  + 0.10 * df["recent_cc_per90"]
  + 0.10 * df["stat_CC"]
  + 0.25 * df["match_over_25_prob"]
  + 0.10 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)
lam_cc = sigmoid((raw_cc - 0.5) * 6) * 3
df["expected_cc_points"] = (lam_cc / 2) * df["prob_plays_60min"]
df.loc[df["position"] != "MID", "expected_cc_points"] = 0.0

In [78]:
cols = [
    "name",
    "team",
    "position",
    "expected_cc_points",
    "cc_per90",
    "recent_cc_per90",
    "stat_CC",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_cc_points", ascending=False).head(30)

,name,team,position,expected_cc_points,cc_per90,recent_cc_per90,stat_CC,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
162,Michael Olise,France,MID,0.675390,0.162338,0.178899,1.0,1.000000,0.541660,0.857143,0.803942
150,Ousmane Dembélé,France,MID,0.496477,0.034722,0.062500,0.2,1.000000,0.725769,0.928571,0.781460
48,Vinícius Júnior,Brazil,MID,0.454481,0.028490,0.049808,0.2,0.796663,0.718813,0.928571,0.873305
230,Julio Enciso,Paraguay,MID,0.379423,0.061350,0.055085,0.4,1.000000,0.227392,0.442857,0.789073
346,Leandro Trossard,Belgium,MID,0.366623,0.055402,0.095941,0.4,0.796663,0.413201,0.442857,0.862233
323,Bruno Guimarães,Brazil,MID,0.362943,0.088235,0.150000,0.6,0.796663,0.227392,0.471429,0.796764
333,Bradley Barcola,France,MID,0.329775,0.048077,0.065657,0.2,1.000000,0.584171,0.642857,0.599648
371,Martin Ødegaard,Norway,MID,0.317972,0.076628,0.072222,0.4,0.796663,0.243005,0.600000,0.745991
251,Bruno Fernandes,Portugal,MID,0.310615,0.060060,0.106996,0.4,0.614339,0.298810,0.714286,0.825673
210,Matías Galarza,Paraguay,MID,0.301371,0.033557,0.043624,0.2,1.000000,0.171433,0.185714,0.804053


In [79]:
raw_sot = (
    0.35 * df["sot_per90"]
  + 0.20 * df["goal_rate"]
  + 0.15 * df["stat_GS"]
  + 0.20 * df["match_over_25_prob"]
  + 0.10 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)
lam_sot = sigmoid((raw_sot - 0.5) * 6) * 3  # expected SoT volume per game
df["expected_sot_points"] = (lam_sot / 2) * df["prob_plays_60min"]
df.loc[~df["position"].isin(["FWD"]), "expected_sot_points"] = 0.0

In [80]:
cols = [
    "name",
    "team",
    "position",
    "expected_sot_points",
    "sot_per90",
    "goal_rate",
    "stat_GS",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_sot_points", ascending=False).head(30)

,name,team,position,expected_sot_points,sot_per90,goal_rate,stat_GS,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
149,Kylian Mbappé,France,FWD,1.086880,0.351852,0.324786,0.857143,1.000000,1.000000,1.000000,0.922048
5,Lionel Messi,Argentina,FWD,1.008044,0.445312,0.415625,1.000000,0.475278,0.943886,0.928571,0.912498
199,Erling Haaland,Norway,FWD,0.859621,0.316667,0.351852,0.714286,0.796663,0.725769,1.000000,0.851601
262,Mikel Oyarzabal,Spain,FWD,0.629257,0.252492,0.252492,0.571429,0.614339,0.725769,0.657143,0.876473
352,Cristiano Ronaldo,Portugal,FWD,0.545762,0.189459,0.162393,0.428571,0.614339,0.591127,0.928571,0.891920
49,Matheus Cunha,Brazil,FWD,0.525831,0.202128,0.242553,0.428571,0.796663,0.633792,0.542857,0.794758
128,Harry Kane,England,FWD,0.500544,0.241525,0.268362,0.714286,0.000000,0.633792,1.000000,0.910706
31,Romelu Lukaku,Belgium,FWD,0.440351,0.107345,0.214689,0.285714,0.796663,0.698717,0.557143,0.802544
65,Jonathan David,Canada,FWD,0.296846,0.200906,0.172205,0.428571,0.183560,0.366672,0.500000,0.885500
320,Lautaro Martínez,Argentina,FWD,0.241714,0.078189,0.078189,0.142857,0.475278,0.692224,0.757143,0.682138


In [81]:
df["expected_qualification_points"] = df["team_qualify_prob"] * 2

In [82]:
cols = [
    "name",
    "team",
    "position",
    "expected_qualification_points",
    "team_qualify_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_qualification_points", ascending=False).head(30)

,name,team,position,expected_qualification_points,team_qualify_prob,price,prob_plays_60min
334,Jean-Philippe Mateta,France,FWD,2.000000,1.000000,0.428571,0.212867
331,William Saliba,France,DEF,2.000000,1.000000,0.257143,0.689645
332,Jules Koundé,France,DEF,2.000000,1.000000,0.271429,0.803128
333,Bradley Barcola,France,MID,2.000000,1.000000,0.642857,0.599648
154,Brice Samba,France,GK,2.000000,1.000000,0.142857,0.206689
158,Manu Koné,France,MID,2.000000,1.000000,0.371429,0.226586
156,Adrien Rabiot,France,MID,2.000000,1.000000,0.414286,0.586244
142,Theo Hernández,France,DEF,2.000000,1.000000,0.214286,0.363809
330,Robin Risser,France,GK,2.000000,1.000000,0.000000,0.189233
153,Mike Maignan,France,GK,2.000000,1.000000,0.214286,0.921262


In [83]:
df.to_csv(OUT_CSV, index=False)

print(f"\nSaved {OUT_CSV} {df.shape[0]} rows x {df.shape[1]} cols")


Saved fantasy_enriched.csv 414 rows x 219 cols


In [84]:
BUDGET = 105.0
MAX_PER_COUNTRY = 4
SQUAD_SIZE = 15
XI_SIZE = 11


def compute_total_expected_points(df):
    df = df.copy()

    cs_value = {"GK": 5, "DEF": 5, "MID": 1, "FWD": 0}
    goal_value = {"GK": 9, "DEF": 7, "MID": 6, "FWD": 5}

    df["goal_pts_value"] = df["position"].map(goal_value)
    df["cs_pts_value"] = df["position"].map(cs_value)

    # appearance: 1pt for playing any, +1 if 60+
    e_appearance = df["prob_plays_any"] * 1 + df["prob_plays_60min"] * 1

    # goal
    e_goal = df["prob_scores"] * df["goal_pts_value"]

    # assist
    e_assist = df["expected_assists"] * 3

    # clean sheet: gated by 60min, value depends on position
    e_cs = df["prob_clean_sheet"] * df["cs_pts_value"]

    # goals conceded penalty: GK and DEF only
    e_gc = df["expected_gc_penalty"].copy()
    e_gc[~df["position"].isin(["GK", "DEF"])] = 0.0

    # saves bonus: GK only, every 3 saves = +1
    e_saves = (df["expected_saves"] / 3).copy()
    e_saves[df["position"] != "GK"] = 0.0

    # penalty save: GK only
    e_pen_save = (df["prob_penalty_save"] * 3).copy()
    e_pen_save[df["position"] != "GK"] = 0.0

    # tackles bonus: MID only, every 3 tackles = +1
    e_tackles = df["expected_tackle_points"].copy()
    e_tackles[df["position"] != "MID"] = 0.0

    # big chances created: MID only, every 2 = +1
    e_cc = df["expected_cc_points"].copy()
    e_cc[df["position"] != "MID"] = 0.0

    # shots on target: FWD only, every 2 = +1
    e_sot = df["expected_sot_points"].copy()
    e_sot[df["position"] != "FWD"] = 0.0

    # yellow card
    e_yc = df["prob_yellow_card"] * -1

    # red card
    e_rc = df["prob_red_card"] * -2

    # own goal
    e_og = df["prob_own_goal"] * -2

    # penalty won
    e_pw = df["prob_pen_won"] * 2

    # penalty committed
    e_pc = df["prob_penalty_committed"] * -1

    # qualification bonus: +2 per player in XI who advances, requires playing
    # captain's quali bonus is NOT doubled per rules
    e_quali = df["expected_qualification_points"] * df["prob_plays_any"]

    # scouting bonus: +2 if differential==1 AND player scores >4 base pts
    # model: use Poisson confidence interval on expected base points
    # base pts = everything except scouting and captain multiplier
    base_ep = (
        e_appearance + e_goal + e_assist + e_cs + e_gc +
        e_saves + e_pen_save + e_tackles + e_cc + e_sot +
        e_yc + e_rc + e_og + e_pw + e_pc + e_quali
    )

    # Poisson 90% CI: lower = ppf(0.05, mu), upper = ppf(0.95, mu)
    mu = base_ep.clip(lower=0.01)
    lower_ci = pd.Series(poisson.ppf(0.05, mu), index=df.index)
    upper_ci = pd.Series(poisson.ppf(0.95, mu), index=df.index)

    # if lower CI > 4: full +2 expected
    # if upper CI < 4: 0
    # if CI straddles 4: scale by P(X>4 | mu) * 2
    p_exceeds_4 = pd.Series(1 - poisson.cdf(4, mu), index=df.index)

    scouting_ep = pd.Series(0.0, index=df.index)
    eligible = df["differential"] == 1
    full_bonus = eligible & (lower_ci > 4)
    no_bonus = eligible & (upper_ci < 4)
    partial = eligible & ~full_bonus & ~no_bonus

    scouting_ep[full_bonus] = 2.0
    scouting_ep[no_bonus] = 0.0
    scouting_ep[partial] = p_exceeds_4[partial] * 2.0

    df["e_appearance"] = e_appearance
    df["e_goal"] = e_goal
    df["e_assist"] = e_assist
    df["e_cs"] = e_cs
    df["e_gc"] = e_gc
    df["e_saves"] = e_saves
    df["e_pen_save"] = e_pen_save
    df["e_tackles"] = e_tackles
    df["e_cc"] = e_cc
    df["e_sot"] = e_sot
    df["e_yc"] = e_yc
    df["e_rc"] = e_rc
    df["e_og"] = e_og
    df["e_pw"] = e_pw
    df["e_pc"] = e_pc
    df["e_quali"] = e_quali
    df["e_scouting"] = scouting_ep
    df["expected_points"] = base_ep + scouting_ep

    return df

In [ ]:

def optimize_squad(df, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):
    idx = df.index.tolist()

    x = LpVariable.dicts("squad", idx, cat="Binary")
    s = LpVariable.dicts("xi", idx, cat="Binary")
    b = LpVariable.dicts("bench", idx, cat="Binary")
    cap = LpVariable.dicts("cap", idx, cat="Binary")

    model = LpProblem("fantasy", LpMaximize)

    # objective: XI points + captain doubling + bench at 0 (bench value is implicit
    # through squad construction — we select best 15 then best 11 starts)
    model += lpSum(
        df.loc[i, "expected_points"] * s[i]
        + df.loc[i, "expected_points"] * cap[i]
        for i in idx
    )

    # squad = 15, xi = 11, bench = 4
    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    # budget
    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    # squad composition: 2 GK, 5 DEF, 5 MID, 3 FWD
    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    # country limit
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country

    # XI formation: 1 GK starts, valid outfield formation
    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    # captain: exactly 1, must be in XI
    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad = [i for i in idx if value(x[i]) > 0.5]
    in_xi = [i for i in idx if value(s[i]) > 0.5]
    on_bench = [i for i in idx if value(b[i]) > 0.5]
    captain = next(i for i in idx if value(cap[i]) > 0.5)

    return {
        "squad": df.loc[in_squad].copy(),
        "xi": df.loc[in_xi].copy(),
        "bench": df.loc[on_bench].copy(),
        "captain": captain,
        "total_ep": value(model.objective),
        "cost": df.loc[in_squad, "price_raw"].sum(),
    }





In [ ]:
def optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY):
    # best possible 11 ignoring bench constraint, just 11 players
    idx = df.index.tolist()

    s = LpVariable.dicts("xi", idx, cat="Binary")
    cap = LpVariable.dicts("cap", idx, cat="Binary")

    model = LpProblem("ideal_xi", LpMaximize)

    model += lpSum(
        df.loc[i, "expected_points"] * s[i]
        + df.loc[i, "expected_points"] * cap[i]
        for i in idx
    )

    model += lpSum(s[i] for i in idx) == 11

    # no budget constraint for ideal XI comparison
    # country limit still applies
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(s[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        return None

    in_xi = [i for i in idx if value(s[i]) > 0.5]
    captain = next(i for i in idx if value(cap[i]) > 0.5)

    return {
        "xi": df.loc[in_xi].copy(),
        "captain": captain,
        "total_ep": value(model.objective),
    }


def print_team(result, ideal=None):
    pos_order = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}
    cap = result["captain"]

    xi = result["xi"].copy()
    xi["_ord"] = xi["position"].map(pos_order)
    xi = xi.sort_values(["_ord", "expected_points"], ascending=[True, False])

    bench = result["bench"].copy()
    bench["_ord"] = bench["position"].map(pos_order)
    bench = bench.sort_values(["_ord", "expected_points"], ascending=[True, False])

    print(f"\nExpected points: {result['total_ep']:.2f}   Cost: ${result['cost']:.1f}M\n")

    print("Starting XI")
    for i, row in xi.iterrows():
        tag = " [C]" if i == cap else ""
        scout = " [SCOUT]" if row.get("differential", 0) == 1 else ""
        print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{scout}")

    print("\nBench")
    for rank, (i, row) in enumerate(bench.iterrows(), 1):
        print(f"  [{rank}] {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}")

    print(f"\nCountry breakdown: {dict(result['squad']['team'].value_counts())}")

    if ideal:
        ideal_xi = ideal["xi"].copy()
        ideal_xi["_ord"] = ideal_xi["position"].map(pos_order)
        ideal_xi = ideal_xi.sort_values(["_ord", "expected_points"], ascending=[True, False])
        ideal_cap = ideal["captain"]

        print(f"\nIdeal XI (no budget constraint, 11 only, EP: {ideal['total_ep']:.2f})")
        for i, row in ideal_xi.iterrows():
            tag = " [C]" if i == ideal_cap else ""
            in_squad = i in result["squad"].index
            flag = "" if in_squad else " [NOT IN SQUAD]"
            print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{flag})




In [87]:
if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)

    df = compute_total_expected_points(df)

    result = optimize_squad(df, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)

    print_team(result, ideal)


Expected points: 104.13   Cost: $6.6M

Starting XI
  GK   Emiliano Martínez            Argentina              $0.2  EP:9.55
  DEF  Dayot Upamecano              France                 $0.3  EP:8.83
  DEF  Achraf Hakimi                Morocco                $0.4  EP:8.15
  DEF  Lisandro Martínez            Argentina              $0.2  EP:7.99
  DEF  Dávinson Sánchez             Colombia               $0.1  EP:7.63 [SCOUT]
  MID  Michael Olise                France                 $0.9  EP:8.79
  MID  Ousmane Dembélé              France                 $0.9  EP:8.54
  MID  Vinícius Júnior              Brazil                 $0.9  EP:8.00
  MID  Alexis Mac Allister          Argentina              $0.4  EP:7.90 [SCOUT]
  FWD  Kylian Mbappé                France                 $1.0  EP:9.79 [C]
  FWD  Lionel Messi                 Argentina              $0.9  EP:9.16

Bench
  [1] GK   Mohamed El Shenawy           Egypt                  $0.0  EP:0.71
  [2] DEF  Mohamed Abdelmonem           E